# LAB 12: NUMERICAL INTEGRATION METHODS (Gaussian-Legendre and Monte-Carlo)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
import math as m

# Below are the definitions/functions we need for Gaussian-Legendre Quadrature. 

We create a function for the Legendre Polynomials

In [2]:
def legendre(n,x):
    if n==0:
        val2 = 1.
        dval2 = 0.
    elif n==1:
        val2 = x
        dval2 = 1.
    else:
        val0 = 1.; val1=x
        for j in range(1,n):
            # recurrence relation for the Legendre polynomials
            val2 = ((2*j+1)*x*val1-j*val0)/(j+1)
            val0, val1 = val1, val2
        # recurrence relation for the first derivative of the Legendre polynomials
        dval2 = n*(val0-x*val1)/(1.-x**2)
    return val2, dval2

Define a function legnewton(n,xold,kmax,tol) for finding the root of the Legendre polynomial of degree  n  for some initial guess  x0  (xold) and with a maximum number of iterations (kmax) and a set value of tolerance (tol).

In [3]:
def legnewton(n, xold, kmax=200, tol=1.e-8):
    for k in range(1,kmax):
        val, dval = legendre(n, xold)
        xnew = xold - val/dval

        xdiff = xnew - xold
        if abs(xdiff/xnew) < tol:
            break

        xold = xnew
    else:
        xnew = None
    return xnew

Define a function legroots(n) which calls the function legnewton(n,xold,kmax,tol) to find all the roots for a Legendre polynomial of degree n.

In [4]:
def legroots(n):
    roots = np.zeros(n)
    npos = n//2
    for i in range(npos):
        xold = np.cos(np.pi*(4*i+3)/(4*n+2)) # generate inital guesses for roots
        root = legnewton(n,xold)
        roots[i] = -root
        roots[-1-i] = root
    return roots

Define a function gauleg_params(n) to find the  xk  (stored in the 1-D array xs) and  ck  parameters (stored in the 1-D array cs).

In [5]:
def gauleg_params(n):
    xs = legroots(n)
    cs = 2/((1-xs**2)*legendre(n,xs)[1]**2) # use derivative of Legendre polynomial
    return xs, cs

Define a function gauleg(f,a,b,n) to perform the Gauss-Legendre quadrature in the interval  [a,b]  and using  n  points.

In [6]:
def gauleg(f,a,b,n):
    xs, cs = gauleg_params(n)
    coeffp = 0.5*(b+a)
    coeffm = 0.5*(b-a)
    ts = coeffp + coeffm*xs
    contribs = cs*f(ts)
    return coeffm*np.sum(contribs)

# Below are the functions we need for the Monte-Carlo Quadrature.

Define a function stats(fs) for evaluating the sample mean and standard deviation given an array of function values fs

In [7]:
def stats(fs):
    n = fs.size
    fbar = np.sum(fs)/n
    fsq = np.sum(fs**2)/n
    varfbar = (fsq - fbar**2)/(n-1)
    return fbar, np.sqrt(varfbar)

Define a function `montecarlo(f,a,b,n)` for approximating the integral between `a` and `b` of a function defined by `f` using `n` points.

In [8]:
def montecarlo(f,a,b,n):
    np.random.seed(314159)
    us = np.random.uniform(a, b, n)
    fs = f(us)
    fbar, err = stats(fs)
    return (b-a)*fbar #i removed the error part of the code.

# PART 1. Comparison of Numerical Integration Methods

We first define our function f(x).
                            $f(x) = e^-{\frac{x^{2}}{2}}$

In [9]:
def Gaussian(x):
    return np.exp(-(x**2)/2)

We find the answer by using the Gaussian-Legendre Quadrature for n = 10 and by using the Monte-Carlo Quadrature for $10$ to $10^5$.

In [10]:
nw = np.array([10, 10**2, 10**3, 10**4, 10**5])
gau = gauleg(Gaussian, -1 , 1, 10)
monte_carlo = [montecarlo(Gaussian, -1, 1, n) for n in nw]

print("Gaussian-Legendre Quadrature for n = 10")
print(f"n = {10}: {gau}")

print("-" * 60)

print("Monte-Carlo Quadrature")
for i in range(len(nw)):
    print(f"n = {nw[i]}: {monte_carlo[i]}")



Gaussian-Legendre Quadrature for n = 10
n = 10: 1.7112487837842938
------------------------------------------------------------
Monte-Carlo Quadrature
n = 10: 1.7030419836408062
n = 100: 1.688507876425818
n = 1000: 1.6978445167939198
n = 10000: 1.7068018574922825
n = 100000: 1.7116179154060005


We solve for the integrate analytically using the sympy package, this is what we did in lab 11.

In [11]:
x = sp.Symbol('x')
definite_res = sp.integrate(sp.exp(-(x**2)/2), (x, -1, 1)) 
print("Result (via analytical method):", definite_res) #this doesnt give a numerical value

print("We evaluate the function above using the math package")
print(" ")
#we will use the math package to obtain the numerical value
analytical = m.sqrt(2)*m.sqrt(m.pi)*m.erf(m.sqrt(2)/2) 
print("Result (via analytical method):", analytical)

Result (via analytical method): sqrt(2)*sqrt(pi)*erf(sqrt(2)/2)
We evaluate the function above using the math package
 
Result (via analytical method): 1.7112487837842978


We also find the difference of the answers we obtained above, Gaussian-Legendre and Monte-Carlo with the analytical result.

In [12]:
gausslegendre_diff = gau - analytical
montecarlo_diff = [(montecarlo(Gaussian, -1, 1, n) - analytical) for n in nw]

print("Gauss-Legendre Quadrature Difference With Analytical:") 
print((f"n = {10}: {gausslegendre_diff}"))
print("-" * 60)
print("Monte-Carlo Quadrature Difference With Analytical")
for i in range(len(nw)):
    print(f"n = {nw[i]}: {montecarlo_diff[i]}")

Gauss-Legendre Quadrature Difference With Analytical:
n = 10: -3.9968028886505635e-15
------------------------------------------------------------
Monte-Carlo Quadrature Difference With Analytical
n = 10: -0.008206800143491622
n = 100: -0.02274090735847989
n = 1000: -0.013404266990378
n = 10000: -0.004446926292015263
n = 100000: 0.00036913162170271185


We see that for the Gaussian-Legendre at n = 10. The difference is in magnitude of $10^{-15}$. Compared to the Monte-Carlo wherein the difference at n = $10^5$ only reached a magnitude of $10^{-4}$.

# PART 2.  Random Number Generators in Python


In [13]:
import random

In [14]:
nb = 10 #number of bins
n = 10**5 #number of samples

In [15]:
samples = np.random.uniform(0,1,n) 
njs, bins = np.histogram(samples, bins =nb, range = (0,1))
#use the historgram to obtain the njs

In [16]:
print("Njs:", njs) #this are the njs
print("Bins:", bins)
print("Samples:", samples)
print("Total No. of samples by adding all elements in nj:", np.sum(njs))

Njs: [10105 10030  9996 10100  9902  9926  9993  9956  9855 10137]
Bins: [0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1. ]
Samples: [0.14006094 0.05291106 0.12624889 ... 0.96262102 0.0364628  0.89778277]
Total No. of samples by adding all elements in nj: 100000


We see that when we add the njs, we obtain the total number of samples. The njs simply mean how many of the samples are in each bin (or interval). An example of one bin is [0, 0.1], and from the samples, there are 10,105 samples in between this bin/interval.

We exepct to find n/nb samples in each bin. And we see from the njs that it is appproximately the equal to our expect value.

In [17]:
expected = n/nb
print("Expected samples in each bin:", expected)

Expected samples in each bin: 10000.0


Below is the "error" (if that is the right term) from each bin (since it is expected that each bin has 10,000 samples).

In [18]:
print(njs-expected)

[ 105.   30.   -4.  100.  -98.  -74.   -7.  -44. -145.  137.]


In [19]:
chisq = (nb/n)*np.sum((njs - expected) ** 2)
print(rf"X^{2}:", chisq)

X^2: 7.880000000000001


Our chi-squred value must be close to our $n_{b}$ - 1 value. Checking the percent error:

In [20]:
degfreedom = nb -1 
percent_error = np.abs(chisq-degfreedom)*100/ degfreedom 
print("Percent Error:", percent_error)

Percent Error: 12.444444444444436


# PART 3. Monte Carlo Quadrature

We define the function: 
$$f(x) = \sqrt{1 - x^{2}}$$

In [21]:
def f(x):
    return np.sqrt(1 - x**2)

We use the Monte-Carlo code we have above to solve the integral for this function from [0,1].

In [22]:
na = np.array(10**np.linspace(1, 7, 7)) #done by trial and error until i found the one w/ difference of less than 10^-4
int_na = [int(n) for n in na]
integ = [montecarlo(f, 0, 1, n) for n in int_na]
print("Result using Monte-Carlo Quadrature")
for i in range(len(na)):
    print(f"n = {na[i]}: {integ[i]}")

Result using Monte-Carlo Quadrature
n = 10.0: 0.7351162524047096
n = 100.0: 0.7589650018382815
n = 1000.0: 0.7751667935638994
n = 10000.0: 0.7800727040237971
n = 100000.0: 0.7852209121917765
n = 1000000.0: 0.7855336841970192
n = 10000000.0: 0.7853817984205982


I used trigonometric substitution to solve this integral analytically. By letting 
$$x = sin\theta$$ 
$$ dx = cos\theta d\theta $$
We obtain
$$ \sqrt{1-x^{2}} = cos\theta$$
And thus the integral becomes: 
$$\int_{0}^{1}  \sqrt{1-x^{2}} dx = \int_{0}^{\pi/2} cos^{2}(x) d\theta$$
 $$ =  \frac{1}{2} \int_{0}^{\pi/2}  (1+cos(2\theta))d\theta  $$
 $$ = \pi/4 $$

In [23]:
analytical_result = np.pi/4
difference = [np.abs((montecarlo(f, 0, 1, n) - analytical_result)) for n in int_na]

print("Monte-Carlo Quadrature Difference With Analytical")
for i in range(len(na)):
    print(f"n = {na[i]}: {difference[i]}")


Monte-Carlo Quadrature Difference With Analytical
n = 10.0: 0.05028191099273871
n = 100.0: 0.02643316155916675
n = 1000.0: 0.010231369833548887
n = 10000.0: 0.005325459373651142
n = 100000.0: 0.0001772512056718023
n = 1000000.0: 0.00013552079957090957
n = 10000000.0: 1.636497685009175e-05


We then took the difference between the analytical result and the results for diferrent n using the Monte-Carlo Quadrature. We see that at n = $10^{7}$, the difference between the analytical result and the Monte-Carlo Quadrature is less than $10^{-4}$.